<a href="https://colab.research.google.com/github/yourusername/visual-thesaurus-llm/blob/main/qlora_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA Fine-Tuning for Large Language Models

This notebook demonstrates how to fine-tune a pre-trained language model using QLoRA (Quantized Low-Rank Adaptation), a memory-efficient fine-tuning technique that combines 4-bit quantization with LoRA.

**What you'll learn:**
- How to set up your environment for QLoRA fine-tuning
- How to load a pre-trained model with 4-bit quantization
- How to configure QLoRA for memory-efficient fine-tuning
- How to prepare a dataset for fine-tuning
- How to train the model with QLoRA
- How to save and load the QLoRA adapter
- How to generate text with your fine-tuned model
- How to compare memory usage between standard LoRA and QLoRA

## 1. Setup

First, let's install the necessary libraries. We'll need the latest versions of transformers, peft, and bitsandbytes for 4-bit quantization support:

In [ ]:
!pip install -q transformers>=4.30.0 datasets peft>=0.4.0 accelerate bitsandbytes>=0.39.0 trl

Now, let's import the necessary libraries and set up memory tracking:

In [ ]:
import torch
import gc
import psutil
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from transformers import Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, prepare_model_for_kbit_training

# Function to track memory usage
def print_memory_usage():
    # CPU memory
    process = psutil.Process()
    cpu_memory_used = process.memory_info().rss / 1024**2  # in MB
    
    # GPU memory if available
    if torch.cuda.is_available():
        gpu_memory_allocated = torch.cuda.memory_allocated() / 1024**2  # in MB
        gpu_memory_reserved = torch.cuda.memory_reserved() / 1024**2  # in MB
        print(f"CPU Memory: {cpu_memory_used:.2f} MB")
        print(f"GPU Memory Allocated: {gpu_memory_allocated:.2f} MB")
        print(f"GPU Memory Reserved: {gpu_memory_reserved:.2f} MB")
    else:
        print(f"CPU Memory: {cpu_memory_used:.2f} MB")
        print("GPU not available")

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print GPU info if available
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
# Print initial memory usage
print("\nInitial memory usage:")
print_memory_usage()

## 2. Load Pre-trained Model with 4-bit Quantization

We'll load a larger model (OPT-1.3B) with 4-bit quantization to demonstrate the memory efficiency of QLoRA.

In [ ]:
model_name = "facebook/opt-1.3b"  # You can try larger models like "facebook/opt-6.7b" or "EleutherAI/pythia-6.9b"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load model in 4-bit precision
    bnb_4bit_use_double_quant=True, # Use double quantization for 4-bit
    bnb_4bit_quant_type="nf4",      # Use NormalFloat 4-bit quantization
    bnb_4bit_compute_dtype=torch.float16  # Compute in float16
)

# Load the pre-trained model with quantization
print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

# Print memory usage after loading the model
print("\nMemory usage after loading the model:")
print_memory_usage()

## 3. Prepare Model for k-bit Training

Before applying LoRA, we need to prepare the model for k-bit training.

In [ ]:
# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)

# Print memory usage after preparing the model
print("\nMemory usage after preparing the model for k-bit training:")
print_memory_usage()

## 4. Configure QLoRA

Now, let's configure LoRA for our 4-bit quantized model.

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,                       # Rank dimension
    lora_alpha=32,              # Alpha parameter
    target_modules=["q_proj", "v_proj"],  # Which modules to apply LoRA to
    lora_dropout=0.05,          # Dropout probability
    bias="none",                # Bias type
    task_type=TaskType.CAUSAL_LM  # Task type
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# Print memory usage after applying LoRA
print("\nMemory usage after applying LoRA:")
print_memory_usage()

## 5. Prepare Dataset

We'll use a small subset of the WikiText dataset for fine-tuning.

In [ ]:
# Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
print(f"Dataset loaded: {len(dataset)} examples")

# Take a small subset for faster training
dataset = dataset.select(range(1000))
print(f"Using {len(dataset)} examples for training")

# Display a sample
print("\nSample text:")
print(dataset[0]['text'][:500])

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Dataset tokenized: {len(tokenized_dataset)} examples")

## 6. Train the Model

Now, let's train the model with QLoRA.

In [ ]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir="./qlora-opt",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
    gradient_accumulation_steps=2,  # Accumulate gradients to simulate larger batch sizes
    fp16=True,  # Use mixed precision training
)

# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Print memory usage before training
print("\nMemory usage before training:")
print_memory_usage()

# Train the model
trainer.train()

# Print memory usage after training
print("\nMemory usage after training:")
print_memory_usage()

## 7. Save and Load the QLoRA Adapter

After training, we can save the LoRA adapter weights, which are much smaller than the full model.

In [ ]:
# Save the LoRA adapter
model.save_pretrained("./qlora-adapter")
print("LoRA adapter saved to ./qlora-adapter")

# Check the size of the saved adapter
!du -sh ./qlora-adapter

In [ ]:
# Clear memory
del model
gc.collect()
torch.cuda.empty_cache()

# Print memory usage after clearing
print("\nMemory usage after clearing:")
print_memory_usage()

# Load a fresh model with 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the LoRA adapter
model = PeftModel.from_pretrained(base_model, "./qlora-adapter")
print("LoRA adapter loaded successfully")

# Print memory usage after loading the adapter
print("\nMemory usage after loading the adapter:")
print_memory_usage()

## 8. Generate Text with the Fine-tuned Model

Now, let's use our fine-tuned model to generate some text.

In [ ]:
# Set the model to evaluation mode
model.eval()

# Function to generate text
def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    
    # Decode the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# Try generating text with different prompts
prompts = [
    "The history of artificial intelligence",
    "In recent years, machine learning has",
    "The most important development in natural language processing is"
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    generated_text = generate_text(prompt)
    print(f"Generated text: {generated_text}")

## 9. Memory Comparison: LoRA vs. QLoRA

Let's compare the memory usage between standard LoRA and QLoRA for the same model.

In [ ]:
# Clear memory
del model
gc.collect()
torch.cuda.empty_cache()

print("Memory usage after clearing:")
print_memory_usage()

# Function to load model with standard LoRA (16-bit)
def load_standard_lora():
    print("\nLoading model with standard LoRA (16-bit)...")
    # Load the base model in 16-bit
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    
    # Apply LoRA to the model
    model = get_peft_model(base_model, lora_config)
    
    print("Standard LoRA model loaded")
    model.print_trainable_parameters()
    
    print("\nMemory usage with standard LoRA:")
    print_memory_usage()
    
    return model

# Function to load model with QLoRA (4-bit)
def load_qlora():
    print("\nLoading model with QLoRA (4-bit)...")
    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    # Load the base model with 4-bit quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
    
    # Prepare the model for k-bit training
    base_model = prepare_model_for_kbit_training(base_model)
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    
    # Apply LoRA to the model
    model = get_peft_model(base_model, lora_config)
    
    print("QLoRA model loaded")
    model.print_trainable_parameters()
    
    print("\nMemory usage with QLoRA:")
    print_memory_usage()
    
    return model

# Load and compare
try:
    # Try to load standard LoRA (may fail if not enough memory)
    standard_lora_model = load_standard_lora()
    del standard_lora_model
    gc.collect()
    torch.cuda.empty_cache()
except Exception as e:
    print(f"Error loading standard LoRA: {e}")
    print("This demonstrates that standard LoRA requires more memory than available.")

# Load QLoRA
qlora_model = load_qlora()

print("\nMemory Comparison Summary:")
print("QLoRA uses significantly less memory than standard LoRA, allowing you to fine-tune larger models on consumer hardware.")

## 10. Advanced Examples

### 10.1 Experiment with Different Quantization Settings

Let's compare NF4 (NormalFloat 4-bit) with INT4 (Integer 4-bit) quantization.

In [ ]:
# Clear memory
del qlora_model
gc.collect()
torch.cuda.empty_cache()

# Function to load model with different quantization settings
def load_with_quantization(quant_type):
    print(f"\nLoading model with {quant_type} quantization...")
    
    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type=quant_type,
        bnb_4bit_compute_dtype=torch.float16
    )
    
    # Load the base model with 4-bit quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
    
    print(f"Model loaded with {quant_type} quantization")
    print(f"\nMemory usage with {quant_type} quantization:")
    print_memory_usage()
    
    return base_model

# Compare NF4 and INT4 quantization
nf4_model = load_with_quantization("nf4")
del nf4_model
gc.collect()
torch.cuda.empty_cache()

int4_model = load_with_quantization("int4")
del int4_model
gc.collect()
torch.cuda.empty_cache()

print("\nQuantization Comparison Summary:")
print("NF4 (NormalFloat 4-bit) is specifically designed for the weight distributions found in language models and typically provides better performance than INT4 (Integer 4-bit) quantization.")

## 11. Conclusion

In this notebook, we've demonstrated how to fine-tune a pre-trained language model using QLoRA. We've covered:

- Setting up the environment for QLoRA fine-tuning
- Loading a pre-trained model with 4-bit quantization
- Configuring QLoRA for memory-efficient fine-tuning
- Preparing a dataset for fine-tuning
- Training the model with QLoRA
- Saving and loading the QLoRA adapter
- Generating text with the fine-tuned model
- Comparing memory usage between standard LoRA and QLoRA
- Experimenting with different quantization settings

QLoRA is a powerful technique for fine-tuning large language models with extremely limited computational resources. By combining 4-bit quantization with LoRA, QLoRA allows you to fine-tune models that would otherwise be too large for consumer hardware.